# reference パイプラインを「移植」ではなく「独立した別モデル」として建てる（`72_`）

## 問題意識

`69_` で reference（`長期定着予測_crmaine_0816.ipynb`、現1位のベース）の技法2つ
（CatBoostネイティブ`text_features`、在籍月数回帰+Platt較正）を**我々の444列パイプラインに移植**し、
4構成すべてPublicで不採用になった。しかし**移植はアンサンブルの最大の価値＝相関の低さを自ら壊す行為**だった。

我々の予測はほぼ全部CatBoost由来（AutoGluonの勝者も`CatBoost_r137_BAG_L2`、単層も CatBoost）で、
非CatBoost成分は TabPFN だけ。reference は構造が根本的に違う:

| | 我々（`54_`/AutoGluon） | reference |
|---|---|---|
| CV | 入社日順 chronological 80/20 | **StratifiedKFold（ランダム）** |
| Test予測 | 全件再学習＋シード平均 | **fold平均（fold bagging）** |
| depth | 4 | **6** |
| 反復 | 560固定 | 3000＋早期停止 |
| 特徴量 | 441列 | 約130列 |
| テキスト | TF-IDF+SVD 15次元×3 | **CatBoostネイティブ text_features + janome** |

実績として TabPFN(top150) は**単体では CatBoost より 0.002 悪い**のに、相関0.9466のおかげで
Public **-0.0053** を出した（[[tabpfn-ensemble-partner]]）。相手の価値は
**単体の強さ × 相関**で決まる。構造がここまで違えば相関はもっと下がるはずで、そこが狙い。

---

## 【重要】reference にあって我々の441列に無い列を発見した

`月次集約` が作る次の3列を、我々は**一度も使っていない**（441列に存在しないことを確認済み）:

- `観測月数` = その社員の月次データの行数
- `退職済み` = 0-23ヶ月に「退職」行があるか
- `最終月に不在籍` = 最終月の状態が「在籍」でないか

実測（本ノートブック第4節で再確認する）:

| | Train | Test |
|---|---|---|
| 観測月数 < 24 | **117名（全員ラベル0）** | **0名** |
| 退職済み = 1 | **129名** | **0名** |
| 最終月に不在籍 = 1 | 137名 | 8名（月23が休職） |

**これは我々が試していない第3の選択肢である。**

| 129名（早期退職者）の扱い | 結果 |
|---|---|
| 学習から除外する | Public **+0.0018 悪化**（`40_` P1_drop_early）。彼らの情報を失う |
| 残すが指示子を与えない（現行441列） | **129名のラベル0を通常の特徴量で説明させている** |
| **残して指示子も与える** | **未検証** |

理屈: Test には該当者がゼロなので、この指示子は **Test では定数**。
モデルは 129名をその定数枝に「逃がす」ことができ、
**残り2,632名（Testに似た母集団）を通常の特徴量で素直に学習できる**。
[[test-set-is-survivor-filtered]] は「129名は Test 集団の予測にも役立つ情報を持っていた」と結論しており、
指示子はその情報を保ったまま汚染だけを外す中間手段になる。

**ただし逆の可能性もある**: 指示子が129名を完全に吸収してしまうと、実質「学習から除外」と同じになり
`40_` P1_drop_early（+0.0018悪化）を再現するかもしれない。**どちらに転ぶかは事前には決まらない**ので、
本ノートブックで両方を測る。

---

## 事前登録（実行前に確定。結果を見てから増やさない）

**2構成だけを比較する。**

| 構成 | 内容 |
|---|---|
| `R_full` | reference をそのまま再現（生存指示子3列を**含む**） |
| `R_nosurv` | 同上から `観測月数`・`退職済み`・`最終月に不在籍` の**3列だけ**を落とす |

差はこの3列だけなので、[[validation-asymmetry]] の言う「単一の事前登録済み変更」に当たる
（探索による勝者選択ではない）。

### 判定

- **採否の一次判定はブレンド曲線の `argmin w < 1.0`**（[[blend-curve-beats-val-margin-gate]]、`71_`の教訓）。
  単体valの足切りは基準点がハードウェアで0.0055動くので使わない
- 相関・MAD（ラベル不使用）も併記する。ノイズ床 MAD 0.02122
- **本ノートブックでは提出ファイルを作らない**

## 再利用する保存済みファイル（本NBで再計算しないもの）

| 用途 | ファイル | 生成元 |
|---|---|---|
| ブレンド相手・相関比較（val 535名）| `..._54_l2_m_interaction_R0_memofix_plus_LM_valpreds.npy` | **`54_`**（8シード平均、単層CatBoost最良 Public 0.515030）|
| 相関・MAD（test 2502名）| `20260816_pool_poolC_weighted.csv` | **AutoGluon 8実行の平均**（`50_`/`51_`/`53_`/`61_`/`62_`×4）|
| 同上 | `20260816_pool_top150_hire_fixed_avg.csv` | **現最良（Public 0.508699）** |

**それ以外はすべて本ノートブックで新規に計算する。**
reference の特徴量・モデルは我々の441列パイプラインを一切使わず、独立に組む。

In [1]:
# janome は reference が形態素解析に使う。CatBoostのネイティブ text_features に渡す前処理。
!pip install -q catboost janome


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 22.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 84.9 MB/s eta 0:00:00:00:0100:01


In [2]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))


Mounted at /content/drive


In [3]:
import datetime, json, re, time, warnings
import numpy as np, pandas as pd
from catboost import CatBoostClassifier, Pool
from janome.tokenizer import Tokenizer
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold

from common.utils.logger import get_logger
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
SEED = 42; seed_everything(seed=SEED)
TARGET_COL, ID_COL = "10年定着ラベル", "社員ID"
pd.set_option("display.max_columns", None)

SCRIPT_NAME = "72_reference_pipeline_standalone"
TODAY = datetime.datetime.now().strftime("%Y%m%d")
logger = get_logger(SCRIPT_NAME, log_dir=str(PROJECT_ROOT/"logs"))
OUTPUT_DIR = PROJECT_ROOT/"data"/"output"/TODAY; OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
logger.info(f"=== [{SCRIPT_NAME}] 開始 ===")

INPUT = PROJECT_ROOT/"data"/"input"
入社時データ_学習 = pd.read_csv(INPUT/"employee_persona_train.csv")
入社時データ_予測 = pd.read_csv(INPUT/"employee_persona_test.csv")
月次データ_学習   = pd.read_csv(INPUT/"employee_monthly_train.csv")
月次データ_予測   = pd.read_csv(INPUT/"employee_monthly_test.csv")
print(入社時データ_学習.shape, 入社時データ_予測.shape)


[2026-08-16 16:29:58] [INFO] === [72_reference_pipeline_standalone] 開始 ===


INFO:72_reference_pipeline_standalone:=== [72_reference_pipeline_standalone] 開始 ===


(2761, 20) (2502, 19)


## 1. 月次集約（reference cell 7 を忠実に再現）

傾き・前半後半差・欠損率・自己学習の分解を含む。
**生存指示子3列（`観測月数`・`退職済み`・`最終月に不在籍`）もここで作る。**

In [4]:
def 学習時間の合計(text):
    if pd.isna(text) or text == "受講なし": return 0.0
    return sum(float(x) for x in re.findall(r"([\d.]+)時間", text))

def 学習テーマの一覧(text):
    if pd.isna(text) or text == "受講なし": return []
    return [n.strip().replace(" ", "") for n in re.findall(r"([^｜：]+)：[\d.]+時間", text)]

def 傾き(値の列):
    有効 = ~np.isnan(値の列)
    if 有効.sum() < 2: return 0.0
    return np.polyfit(np.arange(len(値の列))[有効], 値の列[有効], 1)[0]

等級の数値 = {"G1":1,"G2":2,"G3":3,"G4":4,"G5":5}
役割の数値 = {"メンバー":1,"シニア":2,"エキスパート":3,"リード":4,"シニアエキスパート":4,"マネージャー":5}
行動列 = ["残業時間","有給取得日数","欠勤日数","研修時間","上司との面談実施回数","情報共有件数","在宅勤務日数"]
評価列 = ["360度評価_親和度","360度評価_信頼度","360度評価_主体度","360度評価_学習度","360度評価_共有貢献度"]
評価系列 = [*評価列, "顧客満足度評価","担当プロジェクト数","360度評価者数"]

# 生存指示子（Trainのみ非定数。第4節で分離して検証する）
生存指示子 = ["観測月数", "退職済み", "最終月に不在籍"]

def 月次集約(月次データ):
    月次データ = 月次データ.sort_values(["社員ID","経過月数"]).copy()
    月次データ["学習時間"]   = 月次データ["自己学習（詳細）"].apply(学習時間の合計)
    月次データ["学習テーマ"] = 月次データ["自己学習（詳細）"].apply(学習テーマの一覧)
    月次データ["等級_num"]  = 月次データ["等級"].map(等級の数値)
    月次データ["役割_num"]  = 月次データ["役割"].map(役割の数値)
    行一覧=[]
    for 社員, g in 月次データ.groupby("社員ID", sort=False):
        g = g.reset_index(drop=True); 初月, 最終月 = g.iloc[0], g.iloc[-1]
        行 = {"社員ID":社員,
              "観測月数":len(g),
              "最終月に不在籍":int(最終月["月末在籍状態"]!="在籍"),
              "休職あり":int((g["月末在籍状態"]=="休職").any()),
              "退職済み":int((g["月末在籍状態"]=="退職").any()),
              "最終月の勤務地":最終月["勤務地"], "最終月の職種":最終月["職種"],
              "最終月の部署ID":最終月["部署ID"],
              "部署異動回数":g["部署ID"].nunique()-1, "上司交代回数":g["上司ID"].nunique()-1,
              "役割変化回数":g["役割"].nunique()-1,
              "初月の等級":初月["等級_num"], "最終月の等級":最終月["等級_num"],
              "昇級あり":int(最終月["等級_num"]>初月["等級_num"]),
              "初月の役割":初月["役割_num"], "最終月の役割":最終月["役割_num"],
              "初月給与":初月["月例給与_円"], "最終月給与":最終月["月例給与_円"],
              "給与の伸び率":(最終月["月例給与_円"]-初月["月例給与_円"])/初月["月例給与_円"],
              "学習時間合計":g["学習時間"].sum()}
        テーマ=[t for 一覧 in g["学習テーマ"] for t in 一覧]
        行["学習txt"]="  ".join(テーマ) if テーマ else "なし"
        行["学習テーマ数"]=len(set(テーマ)); 行["学習月数"]=int((g["学習時間"]>0).sum())
        for 列 in 行動列:
            行[f"{列}_平均"]=g[列].mean(); 行[f"{列}_標準偏差"]=g[列].std(); 行[f"{列}_最終月"]=g[列].iloc[-1]
        行["残業時間_傾き"]=傾き(g["残業時間"].to_numpy(dtype=float))
        for 列 in 評価系列:
            行[f"{列}_平均"]=g[列].mean()
            行[f"{列}_直近"]=g[列].dropna().iloc[-1] if g[列].notna().any() else np.nan
            行[f"{列}_欠損率"]=g[列].isna().mean()
        行["360評価_傾き"]=傾き(g[評価列].mean(axis=1).to_numpy(dtype=float))
        行["360評価_更新回数"]=g["360度評価更新フラグ"].sum()
        行["360評価_更新率"]=g["360度評価更新フラグ"].mean()
        前半, 後半 = g[g["経過月数"]<12], g[g["経過月数"]>=12]
        for 列 in ["残業時間","有給取得日数","欠勤日数","情報共有件数","研修時間","月例給与_円"]:
            行[f"{列}_後半前半差"]=後半[列].mean()-前半[列].mean()
        行["360評価_後半前半差"]=(後半[評価列].mean(axis=1).mean()-前半[評価列].mean(axis=1).mean())
        行一覧.append(行)
    return pd.DataFrame(行一覧)

t0=time.time()
月次集約_学習 = 月次集約(月次データ_学習); 月次集約_予測 = 月次集約(月次データ_予測)
print(月次集約_学習.shape, 月次集約_予測.shape, f"({time.time()-t0:.0f}秒)")


(2761, 79) (2502, 79) (54秒)


## 2. 入社時特徴量とメモパース（reference cell 9 / 11 を忠実に再現）

janome で名詞・動詞・形容詞だけ残した「メモtxt」を作り、CatBoost の `text_features` に渡す。
`49_` のパーサー修正（見出し無し書式Bのフォールバック）は **reference には無い** ので、
忠実性を優先してここでも入れない（我々の`54_`との差分要因を増やさないため）。

In [5]:
テキスト列 = ["入社時メモ","上司からのフィードバック","同僚からのフィードバック"]

def 入社時特徴量(入社時データ):
    d = 入社時データ.copy()
    d["入社年"] = pd.to_datetime(d["入社日"]).dt.year
    d["前職職種"] = d["前職職種"].fillna("前職なし")
    d["初期等級_num"] = d["初期等級"].map(等級の数値)
    d["初期役割_num"] = d["初期役割"].map(役割の数値)
    for 列 in テキスト列: d[f"{列}_文字数"] = d[列].str.len()
    return d

入社時_学習 = 入社時特徴量(入社時データ_学習).reset_index(drop=True)
入社時_予測 = 入社時特徴量(入社時データ_予測).reset_index(drop=True)

初期部署の頻度 = pd.concat([入社時_学習["初期部署ID"], 入社時_予測["初期部署ID"]]).value_counts()
入社時_学習["初期部署ID_頻度"] = 入社時_学習["初期部署ID"].map(初期部署の頻度)
入社時_予測["初期部署ID_頻度"] = 入社時_予測["初期部署ID"].map(初期部署の頻度)
最終部署の頻度 = pd.concat([月次集約_学習["最終月の部署ID"], 月次集約_予測["最終月の部署ID"]]).value_counts()
月次集約_学習["最終月の部署ID_頻度"] = 月次集約_学習["最終月の部署ID"].map(最終部署の頻度)
月次集約_予測["最終月の部署ID_頻度"] = 月次集約_予測["最終月の部署ID"].map(最終部署の頻度)

形態素解析器 = Tokenizer(); 残す品詞 = {"名詞","動詞","形容詞"}
def 文書分割(text):
    if pd.isna(text): return ""
    return " ".join(語.base_form for 語 in 形態素解析器.tokenize(text)
                    if 語.part_of_speech.split(",")[0] in 残す品詞)
t0=time.time()
入社時_学習["メモtxt"] = 入社時_学習["入社時メモ"].apply(文書分割)
入社時_予測["メモtxt"] = 入社時_予測["入社時メモ"].apply(文書分割)
print(f"janome分割 完了 ({time.time()-t0:.0f}秒)"); print(入社時_学習["メモtxt"].iloc[0][:90])


janome分割 完了 (44秒)
経歴 大学 卒 人文 教養 中途 入社 前 職 初期 職種 いずれ コーポレート 人物 所見 必要 情報 周囲 開く 認識 丁寧 合わせる 協 働 進める 姿勢 一貫 する 感じる


In [6]:
勤務地一覧 = ["東京","大阪","愛知","福岡","仙台","北海道","その他"]
転居の否定 = r"許容せず|許容しておらず|許容していない|許容しない|希望しておらず|希望せず|希望していない"
在宅の否定 = r"必須条件と(?:は)?していない|不要"

def メモをパース(メモ):
    メモ = メモ.fillna("")
    働き方の節 = メモ.str.extract(r"・勤務地・働き方：(.+?)(?:\n|$)")[0].fillna("")
    志向の節   = メモ.str.extract(r"・キャリア志向：(.+?)(?:\n|$)")[0].fillna("")
    結果 = pd.DataFrame(index=メモ.index)
    def 最初の勤務地(s):
        候補, 位置 = "不明", 10**9
        for 勤務地 in 勤務地一覧:
            i = s.find(勤務地)
            if 0 <= i < 位置: 候補, 位置 = 勤務地, i
        return 候補
    結果["希望勤務地"] = 働き方の節.map(最初の勤務地)
    転居の句 = 働き方の節.str.extract(r"(転居[^、。]*)")[0].fillna("")
    転居NG = 転居の句.str.contains(転居の否定, regex=True)
    転居OK = 転居の句.str.contains("許容") & ~転居NG
    結果["転居許容"] = np.where(転居NG, 0, np.where(転居OK, 1, -1))
    在宅の句 = 働き方の節.str.extract(r"(在宅[^、。]*)")[0].fillna("")
    在宅NG = 在宅の句.str.contains(在宅の否定, regex=True)
    在宅OK = 在宅の句.str.contains("希望") & ~在宅NG
    結果["在宅希望"] = np.where(在宅OK, 1, np.where(在宅NG, 0, -1))
    def 志向の分類(s):
        if not s.strip(): return "不明"
        if ("限定していない" in s or "方向を限定" in s or "限定せず" in s or "特定していない" in s
            or "方向は特定" in s or "限定した志向は確認されていない" in s): return "限定なし"
        for k in ("専門職","管理職","安定"):
            if k in s: return k
        return "不明"
    結果["キャリア志向"] = 志向の節.map(志向の分類)
    return 結果

for データ in (入社時_学習, 入社時_予測):
    p = メモをパース(データ["入社時メモ"])
    for 列 in ["転居許容","在宅希望","希望勤務地","キャリア志向"]: データ[列] = p[列].to_numpy()
for 列 in ["転居許容","在宅希望","キャリア志向"]:
    print(列, dict(pd.concat([入社時_学習[列], 入社時_予測[列]]).value_counts()))


転居許容 {0: np.int64(2960), 1: np.int64(2027), -1: np.int64(276)}
在宅希望 {1: np.int64(2496), 0: np.int64(2491), -1: np.int64(276)}
キャリア志向 {'限定なし': np.int64(1506), '管理職': np.int64(1284), '専門職': np.int64(1230), '安定': np.int64(954), '不明': np.int64(289)}


## 3. 特徴量の組み立て（reference cell 13）

In [7]:
学習データ = 入社時_学習.merge(月次集約_学習, on=ID_COL, how="left")
予測データ = 入社時_予測.merge(月次集約_予測, on=ID_COL, how="left")

カテゴリ列 = ["入社区分","最終学歴","専攻分野","前職職種","採用経路","性別","初期職種","初期勤務地",
          "初期等級","初期役割","最終月の勤務地","最終月の職種","最終月の部署ID","希望勤務地","キャリア志向"]
text列 = ["メモtxt","学習txt"]
除外列 = [ID_COL, "入社日", "初期部署ID", *テキスト列, TARGET_COL]

正解ラベル = 学習データ[TARGET_COL].astype(int)
特徴量     = 学習データ.drop(columns=[c for c in 除外列 if c in 学習データ.columns])
予測用特徴量 = 予測データ.drop(columns=[c for c in 除外列 if c in 予測データ.columns])
for 列 in カテゴリ列:
    特徴量[列] = 特徴量[列].astype(str).fillna("missing")
    予測用特徴量[列] = 予測用特徴量[列].astype(str).fillna("missing")
特徴量 = 特徴量[予測用特徴量.columns]      # 列順を揃える
print(f"特徴量 {特徴量.shape[1]} 列（カテゴリ{len(カテゴリ列)}・テキスト{len(text列)}）")
assert list(特徴量.columns) == list(予測用特徴量.columns)


特徴量 104 列（カテゴリ15・テキスト2）


## 4. 生存指示子3列の実態確認（本ノートブックの主要な検証対象）

In [8]:
print("=== 生存指示子3列の Train/Test 分布 ===")
for 列 in 生存指示子:
    a, b = 学習データ[列], 予測データ[列]
    print(f"  {列:10s} train: {dict(a.value_counts().head(4))}")
    print(f"  {'':10s} test : {dict(b.value_counts().head(4))}")
print()
obs = 学習データ["観測月数"]
print(f"  観測月数<24 の定着率 = {正解ラベル[obs<24].mean():.4f} (n={int((obs<24).sum())})")
print(f"  観測月数=24 の定着率 = {正解ラベル[obs==24].mean():.4f} (n={int((obs==24).sum())})")
print(f"  → Testでは 観測月数<24 が {int((予測データ['観測月数']<24).sum())} 名 = この列はTestで実質定数")
print("\n  ⚠️ この3列は『Testに存在しない部分集団を隔離するダミー』として働きうるが、")
print("     逆に129名を完全に吸収して 40_ P1_drop_early（+0.0018悪化）を再現する可能性もある。")


=== 生存指示子3列の Train/Test 分布 ===
  観測月数       train: {24: np.int64(2644), 23: np.int64(18), 18: np.int64(17), 20: np.int64(17)}
             test : {24: np.int64(2502)}
  退職済み       train: {0: np.int64(2632), 1: np.int64(129)}
             test : {0: np.int64(2502)}
  最終月に不在籍    train: {0: np.int64(2624), 1: np.int64(137)}
             test : {0: np.int64(2494), 1: np.int64(8)}

  観測月数<24 の定着率 = 0.0000 (n=117)
  観測月数=24 の定着率 = 0.5896 (n=2644)
  → Testでは 観測月数<24 が 0 名 = この列はTestで実質定数

  ⚠️ この3列は『Testに存在しない部分集団を隔離するダミー』として働きうるが、
     逆に129名を完全に吸収して 40_ P1_drop_early（+0.0018悪化）を再現する可能性もある。


## 5. 学習と評価

**2つの評価プロトコルを両方回す。**

- **(a) reference式**: 全2,761名の StratifiedKFold OOF。reference が報告している数字と直接比べられる
- **(b) 我々の式**: 入社日順80%(2,208名)で学習 → 生存者535名で評価。
  **`54_`・`63_`・`68_`・`71_` と同じ土俵**なので、ブレンド相手として比較できる

(b) では早期停止に検証535名を使うとリークするので、**2,208名の内側KFoldで早期停止**し、
各fold modelで535名を予測して平均する。

Test予測は reference と同じく **fold平均 × シード平均**。

In [9]:
学習設定 = dict(iterations=3000, learning_rate=0.03, depth=6,
              loss_function="Logloss", eval_metric="Logloss",
              early_stopping_rounds=100, verbose=0, allow_writing_files=False)
シード一覧 = [42, 2024, 7]     # referenceは[42]のみ。「増やすと精度が上がる」と明記されている
FOLD数 = 5                     # referenceは2。同上

# 我々の検証プロトコル(b)用の分割
_s = 入社時データ_学習.sort_values("入社日")
_cut = int(len(_s)*0.8)
学習期間ID = set(_s.iloc[:_cut][ID_COL]); 検証期間ID = set(_s.iloc[_cut:][ID_COL])
早期退職者ID = set(月次データ_学習.loc[月次データ_学習["月末在籍状態"]=="退職", ID_COL].unique())
検証ID = [i for i in _s.iloc[_cut:][ID_COL] if i not in 早期退職者ID]
is学習期間 = 学習データ[ID_COL].isin(学習期間ID).to_numpy()

# ⚠️ 並び順が命。54_/63_/68_/71_ の保存済み val予測は prepare_split が作る
#    「入社日でソートした順」に並んでいる。ここで boolean マスクを使うと
#    学習データ(CSV順)の並びになり、外部予測との突き合わせが全部ズレる。
#    そこで検証IDの順（=入社日順）の位置インデックスを作って .iloc で取る。
_位置 = pd.Series(np.arange(len(学習データ)), index=学習データ[ID_COL])
検証位置 = _位置.loc[検証ID].to_numpy()
assert len(検証位置) == 535, len(検証位置)
print(f"(b) 学習 {is学習期間.sum()}名 / 検証(生存者) {len(検証位置)}名（入社日順）")


def 実行(構成名, 落とす列):
    使う列 = [c for c in 特徴量.columns if c not in 落とす列]
    cat = [c for c in カテゴリ列 if c in 使う列]
    X, Xte = 特徴量[使う列], 予測用特徴量[使う列]
    y = 正解ラベル

    # --- (a) reference式 OOF（全2761名）+ Test予測 ---
    oof_s, test_s = [], []
    for シード in シード一覧:
        oof = np.zeros(len(X)); tp = np.zeros(len(Xte))
        skf = StratifiedKFold(n_splits=FOLD数, shuffle=True, random_state=シード)
        for 学習行, 検証行 in skf.split(X, y):
            m = CatBoostClassifier(random_seed=シード, **学習設定)
            m.fit(Pool(X.iloc[学習行], y.iloc[学習行], cat_features=cat, text_features=text列),
                  eval_set=Pool(X.iloc[検証行], y.iloc[検証行], cat_features=cat, text_features=text列))
            oof[検証行] = m.predict_proba(Pool(X.iloc[検証行], cat_features=cat, text_features=text列))[:,1]
            tp += m.predict_proba(Pool(Xte, cat_features=cat, text_features=text列))[:,1]/FOLD数
        oof_s.append(oof); test_s.append(tp)
    oof_m, test_m = np.mean(oof_s,0), np.mean(test_s,0)

    # --- (b) 我々の式（2208学習 → 生存者535） ---
    Xtr, ytr = X[is学習期間], y[is学習期間]
    val_preds = []
    for シード in シード一覧:
        skf = StratifiedKFold(n_splits=FOLD数, shuffle=True, random_state=シード)
        ps = []
        for 学習行, 停止行 in skf.split(Xtr, ytr):
            m = CatBoostClassifier(random_seed=シード, **学習設定)
            m.fit(Pool(Xtr.iloc[学習行], ytr.iloc[学習行], cat_features=cat, text_features=text列),
                  eval_set=Pool(Xtr.iloc[停止行], ytr.iloc[停止行], cat_features=cat, text_features=text列))
            ps.append(m.predict_proba(Pool(X.iloc[検証位置], cat_features=cat, text_features=text列))[:,1])
        val_preds.append(np.mean(ps,0))
    val_m = np.mean(val_preds,0)

    res = dict(構成=構成名, 列数=len(使う列),
               oof_全体=log_loss(y, oof_m),
               oof_生存者=log_loss(y[~学習データ[ID_COL].isin(早期退職者ID).to_numpy()],
                                oof_m[~学習データ[ID_COL].isin(早期退職者ID).to_numpy()]),
               val_我々=log_loss(y.iloc[検証位置], val_m))
    np.save(OUTPUT_DIR/f"{TODAY}_{SCRIPT_NAME}_{構成名}_valpreds.npy", val_m)
    np.save(OUTPUT_DIR/f"{TODAY}_{SCRIPT_NAME}_{構成名}_testpreds.npy", test_m)
    np.save(OUTPUT_DIR/f"{TODAY}_{SCRIPT_NAME}_{構成名}_oofpreds.npy", oof_m)
    print(f"  {構成名}: 列数{res['列数']} / OOF全体 {res['oof_全体']:.6f} / "
          f"OOF生存者のみ {res['oof_生存者']:.6f} / **val(我々) {res['val_我々']:.6f}**")
    logger.info(str(res))
    return res, val_m, test_m


結果 = {}
for 構成名, 落とす列 in [("R_full", []), ("R_nosurv", 生存指示子)]:
    print(f"\n▶ {構成名}"); t0=time.time()
    r, v, t = 実行(構成名, 落とす列)
    結果[構成名] = dict(res=r, val=v, test=t)
    print(f"    {time.time()-t0:.0f}秒")


(b) 学習 2208名 / 検証(生存者) 535名（入社日順）

▶ R_full
  R_full: 列数104 / OOF全体 0.482097 / OOF生存者のみ 0.503144 / **val(我々) 0.531235**
[2026-08-16 16:53:39] [INFO] {'構成': 'R_full', '列数': 104, 'oof_全体': 0.4820967203956036, 'oof_生存者': 0.5031438217067703, 'val_我々': 0.5312354028499395}


INFO:72_reference_pipeline_standalone:{'構成': 'R_full', '列数': 104, 'oof_全体': 0.4820967203956036, 'oof_生存者': 0.5031438217067703, 'val_我々': 0.5312354028499395}


    1316秒

▶ R_nosurv
  R_nosurv: 列数101 / OOF全体 0.484283 / OOF生存者のみ 0.502701 / **val(我々) 0.528099**
[2026-08-16 17:15:07] [INFO] {'構成': 'R_nosurv', '列数': 101, 'oof_全体': 0.4842829367234619, 'oof_生存者': 0.5027011078965756, 'val_我々': 0.5280985002068872}


INFO:72_reference_pipeline_standalone:{'構成': 'R_nosurv', '列数': 101, 'oof_全体': 0.4842829367234619, 'oof_生存者': 0.5027011078965756, 'val_我々': 0.5280985002068872}


    1288秒


## 6. 判定: ブレンド曲線（`71_`で確立した一次判定）

**【再利用】** ここから先の CatBoost / プールC / 現最良 は保存済みファイルの読み込みであり、
本ノートブックでは学習していない。

In [10]:
def _find(pat):
    h = sorted((PROJECT_ROOT/"data"/"output").glob(pat)); return h[-1] if h else None

p54 = _find("*/*_54_l2_m_interaction_R0_memofix_plus_LM_valpreds.npy")
cb54 = np.load(p54); cb54 = cb54.mean(0) if cb54.ndim==2 else cb54
msg = f"【再利用】54_ CatBoost(単層最良, Public 0.515030) val予測: {p54.name} — 本NBでは学習していない"
print(msg); logger.info(msg)

y_val = 正解ラベル.iloc[検証位置].to_numpy()
CB = log_loss(y_val, cb54)
print(f"\nCatBoost(54_) val = {CB:.6f}")

# 並び順の検算: 54_ の既知の値(0.505477)から外れていたら、行の対応がズレている
assert abs(CB - 0.505477) < 0.002, (
    f"54_のval={CB:.6f} が既知の0.505477と合わない。検証セットの並び順がズレている。"
    " prepare_split は入社日でソートするので、boolean マスクではなく検証位置(.iloc)を使うこと。")
print("  ✅ 並び順の検算OK（54_の既知値と一致）")

ws = np.arange(0, 1.001, 0.05)
print("\n=== ブレンド曲線  f(w) = logloss(w*CatBoost(54_) + (1-w)*72_) ===")
print(f'{"w_cb":>5s} ' + " ".join(f"{k:>10s}" for k in 結果))
curves = {k: np.array([log_loss(y_val, w*cb54 + (1-w)*結果[k]["val"]) for w in ws]) for k in 結果}
for i, w in enumerate(ws):
    print(f"{w:5.2f} " + " ".join(f"{curves[k][i]:10.6f}" for k in 結果))
print()
for k, c in curves.items():
    wbest = ws[np.argmin(c)]
    判定 = ("❌ どんな重みでもCatBoost単体に勝てない" if wbest >= 1.0
          else "⚠️ w*=0（相手単独が最良＝CatBoostが足を引っ張る。要確認）" if wbest <= 0.0
          else "✅ 混ぜる価値あり")
    print(f"  {k:10s} 単体val={結果[k]['res']['val_我々']:.6f} corr={np.corrcoef(結果[k]['val'],cb54)[0,1]:.4f} "
          f"argmin_w={wbest:.2f} f(w*)={c.min():.6f} 改善={c.min()-c[-1]:+.6f}  {判定}")


【再利用】54_ CatBoost(単層最良, Public 0.515030) val予測: 20260815_54_l2_m_interaction_R0_memofix_plus_LM_valpreds.npy — 本NBでは学習していない
[2026-08-16 17:15:08] [INFO] 【再利用】54_ CatBoost(単層最良, Public 0.515030) val予測: 20260815_54_l2_m_interaction_R0_memofix_plus_LM_valpreds.npy — 本NBでは学習していない


INFO:72_reference_pipeline_standalone:【再利用】54_ CatBoost(単層最良, Public 0.515030) val予測: 20260815_54_l2_m_interaction_R0_memofix_plus_LM_valpreds.npy — 本NBでは学習していない



CatBoost(54_) val = 0.505477
  ✅ 並び順の検算OK（54_の既知値と一致）

=== ブレンド曲線  f(w) = logloss(w*CatBoost(54_) + (1-w)*72_) ===
 w_cb     R_full   R_nosurv
 0.00   0.531235   0.528099
 0.05   0.528182   0.525231
 0.10   0.525330   0.522560
 0.15   0.522670   0.520080
 0.20   0.520198   0.517783
 0.25   0.517908   0.515666
 0.30   0.515797   0.513725
 0.35   0.513863   0.511957
 0.40   0.512103   0.510360
 0.45   0.510517   0.508935
 0.50   0.509105   0.507680
 0.55   0.507868   0.506598
 0.60   0.506809   0.505689
 0.65   0.505929   0.504958
 0.70   0.505235   0.504410
 0.75   0.504732   0.504050
 0.80   0.504429   0.503887
 0.85   0.504335   0.503931
 0.90   0.504465   0.504197
 0.95   0.504838   0.504704
 1.00   0.505477   0.505477

  R_full     単体val=0.531235 corr=0.8619 argmin_w=0.85 f(w*)=0.504335 改善=-0.001142  ✅ 混ぜる価値あり
  R_nosurv   単体val=0.528099 corr=0.8653 argmin_w=0.80 f(w*)=0.503887 改善=-0.001590  ✅ 混ぜる価値あり


In [11]:
# ---- Test側: プールC・現最良との距離（ラベル不使用） ----
te_ids = 入社時データ_予測[ID_COL]
def _csv(p): return pd.read_csv(p, header=None, names=[ID_COL,"p"]).set_index(ID_COL).loc[te_ids,"p"].to_numpy()

pc = _find("*/*_pool_poolC_weighted.csv"); cur = _find("*/*_pool_top150_hire_fixed_avg.csv")
poolC = _csv(pc); best = _csv(cur)
for nm, p in [("プールC(AutoGluon8本平均)", pc), ("現最良(Public 0.508699)", cur)]:
    m=f"【再利用】{nm}: {p.name} — 本NBでは再計算していない"; print(m); logger.info(m)

print(f"\n=== Test予測の距離（ノイズ床 MAD 0.02122 / AutoGluonシード間MAD 0.02381）===")
print(f'{"構成":10s} {"corr(プールC)":>13s} {"MAD":>9s} {"corr(現最良)":>13s} {"MAD":>9s} {"平均":>7s}')
for k in 結果:
    t = 結果[k]["test"]
    print(f"  {k:10s} {np.corrcoef(t,poolC)[0,1]:13.4f} {np.abs(t-poolC).mean():9.5f} "
          f"{np.corrcoef(t,best)[0,1]:13.4f} {np.abs(t-best).mean():9.5f} {t.mean():7.4f}")
print(f"\n  参考: TabPFN(top150)は corr(プールC)=0.9657 / MAD=0.0575 で Public -0.0053 を出した")

# 2構成の差＝生存指示子3列だけの効果
d = 結果["R_full"]["test"] - 結果["R_nosurv"]["test"]
print(f"\n=== 生存指示子3列だけの効果（R_full vs R_nosurv）===")
print(f"  val差   {結果['R_full']['res']['val_我々'] - 結果['R_nosurv']['res']['val_我々']:+.6f}"
      f"  (負なら指示子ありが良い)")
print(f"  Test予測 MAD {np.abs(d).mean():.5f}  corr {np.corrcoef(結果['R_full']['test'],結果['R_nosurv']['test'])[0,1]:.5f}")
print(f"  → MAD がノイズ床 0.02122 を超えていれば『本当に別のモデル』")

pd.DataFrame([結果[k]["res"] for k in 結果]).to_csv(
    OUTPUT_DIR/f"{TODAY}_{SCRIPT_NAME}_summary.csv", index=False)


【再利用】プールC(AutoGluon8本平均): 20260816_pool_poolC_weighted.csv — 本NBでは再計算していない
[2026-08-16 17:15:09] [INFO] 【再利用】プールC(AutoGluon8本平均): 20260816_pool_poolC_weighted.csv — 本NBでは再計算していない


INFO:72_reference_pipeline_standalone:【再利用】プールC(AutoGluon8本平均): 20260816_pool_poolC_weighted.csv — 本NBでは再計算していない


【再利用】現最良(Public 0.508699): 20260816_pool_top150_hire_fixed_avg.csv — 本NBでは再計算していない
[2026-08-16 17:15:09] [INFO] 【再利用】現最良(Public 0.508699): 20260816_pool_top150_hire_fixed_avg.csv — 本NBでは再計算していない


INFO:72_reference_pipeline_standalone:【再利用】現最良(Public 0.508699): 20260816_pool_top150_hire_fixed_avg.csv — 本NBでは再計算していない



=== Test予測の距離（ノイズ床 MAD 0.02122 / AutoGluonシード間MAD 0.02381）===
構成            corr(プールC)       MAD     corr(現最良)       MAD      平均
  R_full            0.8780   0.09493        0.8884   0.09033  0.5964
  R_nosurv          0.8819   0.09299        0.8923   0.08838  0.5920

  参考: TabPFN(top150)は corr(プールC)=0.9657 / MAD=0.0575 で Public -0.0053 を出した

=== 生存指示子3列だけの効果（R_full vs R_nosurv）===
  val差   +0.003137  (負なら指示子ありが良い)
  Test予測 MAD 0.01313  corr 0.99738
  → MAD がノイズ床 0.02122 を超えていれば『本当に別のモデル』


## 7. 読み方と次のアクション

### 判定基準（事前登録済み）

1. **`argmin w < 1.0` かどうか**が一次判定。単体valの大小では決めない
   （[[blend-curve-beats-val-margin-gate]]: 基準点がハードウェアで0.0055動く）
2. `corr(プールC)` が **TabPFNの0.9657より低ければ**、構造の違いが本物の多様性になっている証拠
3. `R_full` vs `R_nosurv` の差は**生存指示子3列だけ**の効果。これが大きければ、
   我々の441列に3列足すだけで効く可能性がある（次のノートブックの候補）

### 期待値について（正直な見積もり）

- ブレンドで **-0.003〜-0.006** 程度が現実的な線。1位との差 0.0154 を一撃で埋める手ではない
- ただし [[eda-v7-findings]] で特徴量が閉じ、[[cpu-model-zoo-closed]] でCPUモデルが閉じ、
  重み再調整も -0.003 程度と測れた今、**まだ触っていない大きな軸はこれだけ**

### やらないこと

- **提出ファイルを作らない。** 重みの探索もしない（[[ensemble-oof-overfitting]]）
- reference の在籍月数回帰+Platt較正は**入れていない**。`69_`でPublic不採用が確定しており
  （[[reference-notebook-1st-place-base]]）、ここで再導入すると差分要因が増える